# IL-10 Induced Peptide Model — Colab 版

**自包含 notebook**：由上往下依序執行每個 cell（或直接 Run all）即可。會在 Colab 環境重建 `src/` 套件、安裝套件、寫入內嵌資料、訓練並顯示結果。**資料已內嵌，無需上傳。**

> 執行前請先到 **執行階段 / Runtime → 變更執行階段類型 → 硬體加速器：GPU**。ESM-2 (650M) 在 GPU 上快非常多。


## 0. 環境檢查

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️ 目前是 CPU runtime；ESM-2 650M 會很慢，建議改用 GPU runtime。')


## 1. 安裝套件

（`torch`/`scikit-learn`/`pandas`/`numpy`/`matplotlib`/`tqdm` Colab 已內建，這裡補裝 `fair-esm` 與 `propy3`。）

In [ ]:
!pip -q install fair-esm==2.0.0 "propy3>=1.1.1" "xgboost>=2.1"


## 2. 重建 `src/` 套件

以下 cell 會把專案原始碼寫進 Colab 檔案系統（內容與原 repo 一致）。

In [ ]:
import os
for d in ['src', 'data', 'results', 'artifacts/cache', 'artifacts/models']:
    os.makedirs(d, exist_ok=True)
print('資料夾建立完成')


In [ ]:
%%writefile src/__init__.py



In [ ]:
%%writefile src/data.py
from __future__ import annotations

import json
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import pandas as pd


STANDARD_AAS = set("ACDEFGHIKLMNPQRSTVWY")
MOD_KEYWORDS = ("PHOS", "DEAM", "CITR", "ACET", "MCM")


@dataclass
class DatasetBundle:
    train_df: pd.DataFrame
    test_df: pd.DataFrame
    audit: dict


def _require_columns(df: pd.DataFrame, path: Path) -> None:
    missing = {"Sequence", "FinalLabel"} - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} is missing columns: {sorted(missing)}")


def _parse_sequence(raw_value: object) -> dict:
    raw = str(raw_value).strip().upper()
    left, has_plus, right = raw.partition("+")
    base_seq = re.sub(r"\s+", "", left)
    annotation = right.strip()
    keyword_hits = {key: int(key in raw) for key in MOD_KEYWORDS}
    mod_count = sum(raw.count(key) for key in MOD_KEYWORDS)
    is_standard = bool(base_seq) and all(char in STANDARD_AAS for char in base_seq)
    return {
        "Sequence": raw,
        "BaseSeq": base_seq,
        "HasMod": int(has_plus == "+"),
        "ModPHOS": keyword_hits["PHOS"],
        "ModDEAM": keyword_hits["DEAM"],
        "ModCITR": keyword_hits["CITR"],
        "ModACET": keyword_hits["ACET"],
        "ModMCM": keyword_hits["MCM"],
        "ModCount": mod_count,
        "Annotation": annotation,
        "IsStandardBaseSeq": int(is_standard),
        "SeqLength": len(base_seq),
        "LogLength": math.log1p(len(base_seq)),
    }


def _prepare_dataframe(path: Path) -> tuple[pd.DataFrame, dict]:
    df = pd.read_csv(path)
    _require_columns(df, path)
    parsed = pd.DataFrame([_parse_sequence(value) for value in df["Sequence"]])
    merged = parsed.copy()
    merged["FinalLabel"] = df["FinalLabel"].astype(int).to_numpy()
    valid = merged["IsStandardBaseSeq"].astype(bool)
    filtered = merged.loc[valid].reset_index(drop=True)

    lengths = filtered["SeqLength"]
    audit = {
        "rows": int(len(df)),
        "valid_rows": int(len(filtered)),
        "invalid_rows": int((~valid).sum()),
        "positive": int((filtered["FinalLabel"] == 1).sum()),
        "negative": int((filtered["FinalLabel"] == 0).sum()),
        "positive_ratio": float(filtered["FinalLabel"].mean()),
        "modified_rows": int(filtered["HasMod"].sum()),
        "unique_raw_sequences": int(filtered["Sequence"].nunique()),
        "unique_base_sequences": int(filtered["BaseSeq"].nunique()),
        "duplicate_raw_sequences": int(filtered["Sequence"].duplicated().sum()),
        "duplicate_base_sequences": int(filtered["BaseSeq"].duplicated().sum()),
        "length_min": int(lengths.min()),
        "length_median": float(lengths.median()),
        "length_max": int(lengths.max()),
    }
    return filtered, audit


def _overlap_summary(train_df: pd.DataFrame, test_df: pd.DataFrame) -> dict:
    train_raw = set(train_df["Sequence"])
    test_raw = set(test_df["Sequence"])
    train_base = set(train_df["BaseSeq"])
    test_base = set(test_df["BaseSeq"])
    raw_overlap = sorted(train_raw & test_raw)
    base_overlap = sorted(train_base & test_base)
    return {
        "raw_overlap_count": len(raw_overlap),
        "base_overlap_count": len(base_overlap),
        "raw_overlap_examples": raw_overlap[:10],
        "base_overlap_examples": base_overlap[:10],
    }


def load_datasets(train_path: Path, test_path: Path) -> DatasetBundle:
    train_df, train_audit = _prepare_dataframe(train_path)
    test_df, test_audit = _prepare_dataframe(test_path)
    audit = {
        "train": train_audit,
        "test": test_audit,
        "overlap": _overlap_summary(train_df, test_df),
        "mod_keywords": list(MOD_KEYWORDS),
    }
    return DatasetBundle(train_df=train_df, test_df=test_df, audit=audit)


def write_audit(audit: dict, path: Path) -> None:
    path.write_text(json.dumps(audit, indent=2), encoding="utf-8")


def modification_feature_columns() -> list[str]:
    return [
        "HasMod",
        "ModPHOS",
        "ModDEAM",
        "ModCITR",
        "ModACET",
        "ModMCM",
        "ModCount",
        "SeqLength",
        "LogLength",
    ]


def dataframe_to_mod_array(df: pd.DataFrame) -> pd.DataFrame:
    return df.loc[:, modification_feature_columns()].copy()



In [ ]:
%%writefile src/features.py
from __future__ import annotations

import os
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_selection import VarianceThreshold

from .data import STANDARD_AAS, dataframe_to_mod_array


AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_INDEX = {aa: idx for idx, aa in enumerate(AA_LIST)}


@dataclass
class EmbeddingFeatures:
    mean_embeddings: np.ndarray
    max_embeddings: np.ndarray


@dataclass
class HandcraftedFeatureSet:
    features: np.ndarray
    feature_names: list[str]


@dataclass
class FeatureSelectionResult:
    train_features: np.ndarray
    test_features: np.ndarray
    feature_names: list[str]
    selection_summary: dict


class ESMEmbedder:
    def __init__(self, model_name: str, batch_size: int) -> None:
        self.model_name = model_name
        self.batch_size = batch_size
        self._model = None
        self._alphabet = None
        self._device = None
        self._layer_index = int(model_name.split("_t")[1].split("_")[0])

    @property
    def device(self) -> str:
        if self._device is None:
            import torch

            self._device = "cuda" if torch.cuda.is_available() else "cpu"
        return self._device

    def _load_model(self) -> tuple[object, object]:
        if self._model is None or self._alphabet is None:
            import esm

            model_loader = getattr(esm.pretrained, self.model_name)
            model, alphabet = model_loader()
            model = model.to(self.device).eval()
            self._model = model
            self._alphabet = alphabet
        return self._model, self._alphabet

    def _embed_sequences(self, sequences: list[str]) -> EmbeddingFeatures:
        import torch
        from tqdm import tqdm

        model, alphabet = self._load_model()
        batch_converter = alphabet.get_batch_converter()
        mean_vectors: list[np.ndarray] = []
        max_vectors: list[np.ndarray] = []

        for start in tqdm(range(0, len(sequences), self.batch_size), desc="ESM2", ncols=80):
            batch = sequences[start : start + self.batch_size]
            entries = [(f"seq_{idx}", sequence) for idx, sequence in enumerate(batch)]
            _, _, tokens = batch_converter(entries)
            tokens = tokens.to(self.device, non_blocking=self.device == "cuda")
            with torch.no_grad():
                outputs = model(tokens, repr_layers=[self._layer_index], return_contacts=False)

            reps = outputs["representations"][self._layer_index]
            for row_index, sequence in enumerate(batch):
                residue_reps = reps[row_index, 1 : len(sequence) + 1]
                mean_vectors.append(residue_reps.mean(dim=0).detach().cpu().numpy())
                max_vectors.append(residue_reps.max(dim=0).values.detach().cpu().numpy())

        return EmbeddingFeatures(
            mean_embeddings=np.asarray(mean_vectors, dtype=np.float32),
            max_embeddings=np.asarray(max_vectors, dtype=np.float32),
        )

    def embed_uncached(self, sequences: list[str]) -> EmbeddingFeatures:
        return self._embed_sequences(sequences)

    def embed(self, sequences: list[str], cache_path: Path) -> EmbeddingFeatures:
        if cache_path.exists():
            cached = np.load(cache_path, allow_pickle=True)
            cached_sequences = cached["sequences"].tolist()
            cached_model_name = str(cached["model_name"].item()) if "model_name" in cached else ""
            if cached_model_name == self.model_name and list(cached_sequences) == list(sequences):
                return EmbeddingFeatures(
                    mean_embeddings=cached["mean_embeddings"].astype(np.float32),
                    max_embeddings=cached["max_embeddings"].astype(np.float32),
                )

        features = self._embed_sequences(sequences)
        np.savez_compressed(
            cache_path,
            model_name=np.asarray(self.model_name),
            sequences=np.asarray(sequences, dtype=object),
            mean_embeddings=features.mean_embeddings,
            max_embeddings=features.max_embeddings,
        )
        return features


def build_esm_feature_views(df: pd.DataFrame, embeddings: EmbeddingFeatures) -> dict[str, np.ndarray]:
    mod_array = dataframe_to_mod_array(df).to_numpy(dtype=np.float32)
    mean_view = np.concatenate([embeddings.mean_embeddings, mod_array], axis=1)
    max_view = np.concatenate([embeddings.max_embeddings, mod_array], axis=1)
    concat_view = np.concatenate([embeddings.mean_embeddings, embeddings.max_embeddings, mod_array], axis=1)
    return {
        "mean": mean_view.astype(np.float32),
        "max": max_view.astype(np.float32),
        "concat": concat_view.astype(np.float32),
    }


def _aac_vector(sequence: str) -> np.ndarray:
    length = len(sequence)
    counts = np.zeros(len(AA_LIST), dtype=np.float32)
    for aa in sequence:
        counts[AA_TO_INDEX[aa]] += 1.0
    return counts / max(length, 1)


def _dpc_vector(sequence: str) -> np.ndarray:
    vector = np.zeros(len(AA_LIST) ** 2, dtype=np.float32)
    if len(sequence) < 2:
        return vector
    for left, right in zip(sequence[:-1], sequence[1:]):
        vector[AA_TO_INDEX[left] * len(AA_LIST) + AA_TO_INDEX[right]] += 1.0
    return vector / (len(sequence) - 1)


def _aaindex_lookup() -> dict[str, dict[str, float]]:
    from propy.AAIndex import _aaindex, init
    import propy

    aaindex_path = os.path.join(os.path.dirname(propy.__file__), "aaindex")
    init(path=aaindex_path)
    lookup: dict[str, dict[str, float]] = {}
    for name, entry in _aaindex.items():
        index = getattr(entry, "index", None)
        if index is None:
            continue
        if all(aa in index and index[aa] is not None for aa in AA_LIST):
            lookup[name] = {aa: float(index[aa]) for aa in AA_LIST}
    return lookup


def build_handcrafted_features(df: pd.DataFrame, cache_path: Path) -> HandcraftedFeatureSet:
    sequences = df["BaseSeq"].tolist()
    if cache_path.exists():
        cached = np.load(cache_path, allow_pickle=True)
        cached_sequences = cached["sequences"].tolist() if "sequences" in cached else []
        if list(cached_sequences) == list(sequences):
            return HandcraftedFeatureSet(
                features=cached["features"].astype(np.float32),
                feature_names=cached["feature_names"].tolist(),
            )

    mod_frame = dataframe_to_mod_array(df)
    aac_matrix = np.vstack([_aac_vector(sequence) for sequence in sequences]).astype(np.float32)
    dpc_matrix = np.vstack([_dpc_vector(sequence) for sequence in sequences]).astype(np.float32)
    aaindex_lookup = _aaindex_lookup()
    aaindex_names = sorted(aaindex_lookup.keys())

    aaindex_rows = []
    for sequence in sequences:
        aaindex_rows.append(
            [float(np.mean([aaindex_lookup[name][aa] for aa in sequence])) for name in aaindex_names]
        )
    aaindex_matrix = np.asarray(aaindex_rows, dtype=np.float32)
    mod_matrix = mod_frame.to_numpy(dtype=np.float32)

    feature_names = (
        [f"AAC_{aa}" for aa in AA_LIST]
        + [f"DPC_{left}{right}" for left in AA_LIST for right in AA_LIST]
        + [f"AAINDEX_{name}" for name in aaindex_names]
        + list(mod_frame.columns)
    )
    features = np.concatenate([aac_matrix, dpc_matrix, aaindex_matrix, mod_matrix], axis=1).astype(np.float32)
    np.savez_compressed(
        cache_path,
        sequences=np.asarray(sequences, dtype=object),
        features=features,
        feature_names=np.asarray(feature_names, dtype=object),
    )
    return HandcraftedFeatureSet(features=features, feature_names=feature_names)


def select_handcrafted_features(
    train_features: np.ndarray,
    test_features: np.ndarray,
    y_train: np.ndarray,
    feature_names: list[str],
    seed: int,
) -> FeatureSelectionResult:
    from xgboost import XGBClassifier

    mask = np.ones(len(feature_names), dtype=bool)
    summary: dict[str, int | float] = {"initial_features": int(len(feature_names))}

    variance_selector = VarianceThreshold(threshold=1e-5)
    variance_selector.fit(train_features)
    variance_mask = variance_selector.get_support()
    mask &= variance_mask
    summary["removed_low_variance"] = int((~variance_mask).sum())

    selected_names = [name for name, keep in zip(feature_names, mask) if keep]
    selected_train = train_features[:, mask]
    selected_test = test_features[:, mask]

    aaindex_positions = [idx for idx, name in enumerate(selected_names) if name.startswith("AAINDEX_")]
    corr_keep = np.ones(selected_train.shape[1], dtype=bool)
    if len(aaindex_positions) > 1:
        aaindex_corr = np.corrcoef(selected_train[:, aaindex_positions].T)
        upper = np.triu(np.abs(aaindex_corr), k=1)
        drop_indices: set[int] = set()
        for col in range(upper.shape[1]):
            if col in drop_indices:
                continue
            correlated = np.where(upper[:col, col] > 0.95)[0]
            drop_indices.update(int(index) for index in correlated)
        for local_index in drop_indices:
            corr_keep[aaindex_positions[local_index]] = False
        summary["removed_correlated_aaindex"] = int(len(drop_indices))
    else:
        summary["removed_correlated_aaindex"] = 0

    selected_names = [name for name, keep in zip(selected_names, corr_keep) if keep]
    selected_train = selected_train[:, corr_keep]
    selected_test = selected_test[:, corr_keep]

    scale_pos = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    importance_model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        min_child_weight=3,
        subsample=0.85,
        colsample_bytree=0.7,
        reg_alpha=0.1,
        reg_lambda=8.0,
        scale_pos_weight=scale_pos,
        tree_method="hist",
        device="cpu",
        random_state=seed,
        n_jobs=-1,
    )
    importance_model.fit(selected_train, y_train, verbose=False)
    importances = importance_model.feature_importances_
    importance_threshold = float(importances.mean())
    importance_mask = importances >= importance_threshold
    selected_names = [name for name, keep in zip(selected_names, importance_mask) if keep]
    selected_train = selected_train[:, importance_mask]
    selected_test = selected_test[:, importance_mask]
    summary["removed_low_importance"] = int((~importance_mask).sum())
    summary["final_features"] = int(len(selected_names))
    summary["importance_threshold"] = importance_threshold

    return FeatureSelectionResult(
        train_features=selected_train.astype(np.float32),
        test_features=selected_test.astype(np.float32),
        feature_names=selected_names,
        selection_summary=summary,
    )


In [ ]:
%%writefile src/model.py
from __future__ import annotations

import math
import pickle
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier


THRESHOLD_LABELS = {
    "default_0p5": "Default threshold 0.5",
    "best_mcc_on_train_oof": "Best MCC threshold from train OOF",
    "best_f2_on_train_oof": "Best F2 threshold from train OOF",
}


@dataclass
class VariantRunResult:
    variant_name: str
    variant_label: str
    train_probabilities: np.ndarray
    test_probabilities: np.ndarray
    train_aux: dict[str, np.ndarray]
    test_aux: dict[str, np.ndarray]
    fold_summary: pd.DataFrame
    model_bundle: dict


@dataclass(frozen=True)
class ESMStackingCandidate:
    name: str
    label: str
    view_names: tuple[str, ...]
    param_overrides: dict[str, dict[str, float]]


def _safe_n_splits(y: np.ndarray, desired: int) -> int:
    class_counts = np.bincount(y.astype(int))
    positive_min = int(class_counts[class_counts > 0].min())
    return max(2, min(desired, positive_min))


def _metric_row(
    variant_name: str,
    variant_label: str,
    evaluation_split: str,
    threshold_key: str,
    y_true: np.ndarray,
    y_prob: np.ndarray,
    threshold: float,
) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else math.nan
    sensitivity = tp / (tp + fn) if (tp + fn) else math.nan
    return {
        "VariantName": variant_name,
        "VariantLabel": variant_label,
        "EvaluationSplit": evaluation_split,
        "ThresholdKey": threshold_key,
        "ThresholdLabel": THRESHOLD_LABELS[threshold_key],
        "ThresholdValue": threshold,
        "RocAuc": roc_auc_score(y_true, y_prob),
        "PrecisionRecallAuc": average_precision_score(y_true, y_prob),
        "MatthewsCorrcoef": matthews_corrcoef(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "RecallSensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
        "F1Score": f1_score(y_true, y_pred, zero_division=0),
        "F2Score": fbeta_score(y_true, y_pred, beta=2, zero_division=0),
        "TrueNegative": int(tn),
        "FalsePositive": int(fp),
        "FalseNegative": int(fn),
        "TruePositive": int(tp),
    }


def select_threshold(y_true: np.ndarray, y_prob: np.ndarray, objective: str) -> float:
    candidates = np.unique(np.clip(y_prob, 0.01, 0.99))
    best_threshold = 0.5
    best_score = -np.inf
    for threshold in candidates:
        y_pred = (y_prob >= threshold).astype(int)
        if objective == "mcc":
            score = matthews_corrcoef(y_true, y_pred)
        elif objective == "f2":
            score = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        else:
            raise ValueError(f"Unsupported threshold objective: {objective}")
        if score > best_score:
            best_score = score
            best_threshold = float(threshold)
    return best_threshold


def build_metric_tables(
    variant_name: str,
    variant_label: str,
    y_train: np.ndarray,
    y_test: np.ndarray,
    train_prob: np.ndarray,
    test_prob: np.ndarray,
) -> tuple[pd.DataFrame, dict[str, float]]:
    thresholds = {
        "default_0p5": 0.5,
        "best_mcc_on_train_oof": select_threshold(y_train, train_prob, "mcc"),
        "best_f2_on_train_oof": select_threshold(y_train, train_prob, "f2"),
    }
    rows = []
    for split_name, labels, probabilities in (
        ("TrainOOF", y_train, train_prob),
        ("IndependentTest", y_test, test_prob),
    ):
        for threshold_key, threshold_value in thresholds.items():
            rows.append(
                _metric_row(
                    variant_name=variant_name,
                    variant_label=variant_label,
                    evaluation_split=split_name,
                    threshold_key=threshold_key,
                    y_true=labels,
                    y_prob=probabilities,
                    threshold=threshold_value,
                )
            )
    return pd.DataFrame(rows), thresholds


def _xgb_params(scale_pos_weight: float, max_depth: int, learning_rate: float, seed: int, with_early_stopping: bool) -> dict:
    params = {
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "n_estimators": 1200,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "min_child_weight": 3,
        "subsample": 0.85,
        "colsample_bytree": 0.7,
        "reg_alpha": 0.1,
        "reg_lambda": 8.0,
        "gamma": 0.1,
        "scale_pos_weight": scale_pos_weight,
        "tree_method": "hist",
        "device": "cpu",
        "random_state": seed,
        "n_jobs": -1,
    }
    if with_early_stopping:
        params["early_stopping_rounds"] = 80
    return params


def train_single_xgb_variant(
    variant_name: str,
    variant_label: str,
    train_features: np.ndarray,
    test_features: np.ndarray,
    y_train: np.ndarray,
    seed: int,
    n_splits: int,
    feature_names: list[str],
) -> VariantRunResult:
    oof_prob = np.zeros(len(y_train), dtype=np.float32)
    best_iterations: list[int] = []
    fold_rows: list[dict] = []
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for fold_index, (train_idx, valid_idx) in enumerate(cv.split(train_features, y_train), start=1):
        y_fold_train = y_train[train_idx]
        scale_pos = float((y_fold_train == 0).sum() / max((y_fold_train == 1).sum(), 1))
        model = XGBClassifier(
            **_xgb_params(
                scale_pos_weight=scale_pos,
                max_depth=5,
                learning_rate=0.04,
                seed=seed,
                with_early_stopping=True,
            )
        )
        model.fit(
            train_features[train_idx],
            y_fold_train,
            eval_set=[(train_features[valid_idx], y_train[valid_idx])],
            verbose=False,
        )
        fold_prob = model.predict_proba(train_features[valid_idx])[:, 1]
        oof_prob[valid_idx] = fold_prob
        best_iteration = int(getattr(model, "best_iteration", model.n_estimators - 1)) + 1
        best_iterations.append(best_iteration)
        fold_rows.append(
            {
                "FoldIndex": fold_index,
                "BestIteration": best_iteration,
                "ValidationRocAuc": roc_auc_score(y_train[valid_idx], fold_prob),
                "ValidationPrecisionRecallAuc": average_precision_score(y_train[valid_idx], fold_prob),
            }
        )

    full_scale_pos = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    final_model = XGBClassifier(
        **_xgb_params(
            scale_pos_weight=full_scale_pos,
            max_depth=5,
            learning_rate=0.04,
            seed=seed,
            with_early_stopping=False,
        )
    )
    final_model.set_params(n_estimators=int(round(np.mean(best_iterations))))
    final_model.fit(train_features, y_train, verbose=False)
    test_prob = final_model.predict_proba(test_features)[:, 1]

    return VariantRunResult(
        variant_name=variant_name,
        variant_label=variant_label,
        train_probabilities=oof_prob,
        test_probabilities=test_prob,
        train_aux={},
        test_aux={},
        fold_summary=pd.DataFrame(fold_rows),
        model_bundle={
            "model_type": "single_xgboost",
            "feature_names": feature_names,
            "model": final_model,
            "mean_best_iteration": float(np.mean(best_iterations)),
        },
    )


def _stack_base_params(scale_pos_weight: float, variant: str, seed: int, with_early_stopping: bool) -> dict:
    base_params = {
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "n_estimators": 1200,
        "learning_rate": 0.03,
        "subsample": 0.85,
        "colsample_bytree": 0.6,
        "min_child_weight": 3,
        "reg_alpha": 0.1,
        "reg_lambda": 8.0,
        "scale_pos_weight": scale_pos_weight,
        "tree_method": "hist",
        "device": "cpu",
        "random_state": seed,
        "n_jobs": -1,
    }
    per_view = {
        "mean": {"max_depth": 3, "gamma": 0.05},
        "max": {"max_depth": 4, "gamma": 0.15},
        "concat": {
            "max_depth": 4,
            "gamma": 0.1,
            "learning_rate": 0.02,
            "subsample": 0.8,
            "colsample_bytree": 0.45,
            "min_child_weight": 5,
        },
    }
    if variant not in per_view:
        raise ValueError(f"Unknown stack base variant: {variant}")
    params = dict(base_params)
    params.update(per_view[variant])
    if with_early_stopping:
        params["early_stopping_rounds"] = 80
    return params


def _fit_stack_base_model(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_valid: np.ndarray,
    y_valid: np.ndarray,
    variant: str,
    seed: int,
    param_overrides: dict[str, float] | None = None,
) -> tuple[XGBClassifier, np.ndarray, int]:
    scale_pos = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    model = XGBClassifier(**_stack_base_params(scale_pos, variant, seed, with_early_stopping=True))
    if param_overrides:
        model.set_params(**param_overrides)
    model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)
    valid_prob = model.predict_proba(x_valid)[:, 1]
    best_iter = int(getattr(model, "best_iteration", model.n_estimators - 1)) + 1
    return model, valid_prob, best_iter


def esm_tuning_candidates() -> list[ESMStackingCandidate]:
    return [
        ESMStackingCandidate(
            name="baseline_mean_max",
            label="Mean + max pooling stack",
            view_names=("mean", "max"),
            param_overrides={},
        ),
        ESMStackingCandidate(
            name="mean_max_concat",
            label="Mean + max + concat stack",
            view_names=("mean", "max", "concat"),
            param_overrides={},
        ),
        ESMStackingCandidate(
            name="regularized_mean_max",
            label="Regularized mean + max stack",
            view_names=("mean", "max"),
            param_overrides={
                "mean": {"learning_rate": 0.02, "max_depth": 2, "colsample_bytree": 0.45, "gamma": 0.1},
                "max": {"learning_rate": 0.02, "max_depth": 3, "colsample_bytree": 0.5, "gamma": 0.2},
            },
        ),
    ]


def _view_probability_name(view_name: str) -> str:
    mapping = {
        "mean": "MeanEmbeddingProbability",
        "max": "MaxEmbeddingProbability",
        "concat": "ConcatEmbeddingProbability",
    }
    return mapping[view_name]


def train_esm_stacking_variant(
    variant_name: str,
    variant_label: str,
    train_views: dict[str, np.ndarray],
    test_views: dict[str, np.ndarray],
    y_train: np.ndarray,
    seed: int,
    n_splits: int,
    candidate: ESMStackingCandidate | None = None,
) -> VariantRunResult:
    candidate = candidate or esm_tuning_candidates()[0]
    view_names = list(candidate.view_names)
    oof_base = np.zeros((len(y_train), len(view_names)), dtype=np.float32)
    oof_stack = np.zeros(len(y_train), dtype=np.float32)
    best_iterations = {view_name: [] for view_name in view_names}
    fold_rows: list[dict] = []
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    for fold_index, (train_idx, valid_idx) in enumerate(cv.split(train_views["mean"], y_train), start=1):
        y_fold_train = y_train[train_idx]
        outer_valid_base = np.zeros((len(valid_idx), len(view_names)), dtype=np.float32)
        inner_train_views = {name: values[train_idx] for name, values in train_views.items()}
        inner_splits = _safe_n_splits(y_fold_train, max(3, n_splits - 1))
        inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=seed + fold_index)
        inner_oof_base = np.zeros((len(train_idx), len(view_names)), dtype=np.float32)

        for inner_train_idx, inner_valid_idx in inner_cv.split(inner_train_views["mean"], y_fold_train):
            for column_index, view_name in enumerate(view_names):
                _, inner_valid_prob, _ = _fit_stack_base_model(
                    x_train=inner_train_views[view_name][inner_train_idx],
                    y_train=y_fold_train[inner_train_idx],
                    x_valid=inner_train_views[view_name][inner_valid_idx],
                    y_valid=y_fold_train[inner_valid_idx],
                    variant=view_name,
                    seed=seed,
                    param_overrides=candidate.param_overrides.get(view_name),
                )
                inner_oof_base[inner_valid_idx, column_index] = inner_valid_prob

        for column_index, view_name in enumerate(view_names):
            _, valid_prob, best_iter = _fit_stack_base_model(
                x_train=train_views[view_name][train_idx],
                y_train=y_fold_train,
                x_valid=train_views[view_name][valid_idx],
                y_valid=y_train[valid_idx],
                variant=view_name,
                seed=seed,
                param_overrides=candidate.param_overrides.get(view_name),
            )
            oof_base[valid_idx, column_index] = valid_prob
            outer_valid_base[:, column_index] = valid_prob
            best_iterations[view_name].append(best_iter)

        meta_fold = LogisticRegression(class_weight="balanced", max_iter=4000, random_state=seed)
        meta_fold.fit(inner_oof_base, y_fold_train)
        stack_valid_prob = meta_fold.predict_proba(outer_valid_base)[:, 1]
        oof_stack[valid_idx] = stack_valid_prob

        fold_rows.append(
            {
                "FoldIndex": fold_index,
                "StackPrAuc": average_precision_score(y_train[valid_idx], stack_valid_prob),
                "StackRocAuc": roc_auc_score(y_train[valid_idx], stack_valid_prob),
                "CandidateName": candidate.name,
            }
        )
        for column_index, view_name in enumerate(view_names):
            fold_rows[-1][f"{view_name.title()}BestIteration"] = best_iterations[view_name][-1]
            fold_rows[-1][f"{view_name.title()}PrAuc"] = average_precision_score(
                y_train[valid_idx], outer_valid_base[:, column_index]
            )

    meta_model = LogisticRegression(class_weight="balanced", max_iter=4000, random_state=seed)
    meta_model.fit(oof_base, y_train)

    full_models: dict[str, XGBClassifier] = {}
    test_base = np.zeros((test_views["mean"].shape[0], len(view_names)), dtype=np.float32)
    full_scale_pos = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
    for column_index, view_name in enumerate(view_names):
        params = _stack_base_params(full_scale_pos, view_name, seed, with_early_stopping=False)
        params.update(candidate.param_overrides.get(view_name, {}))
        params["n_estimators"] = int(round(np.mean(best_iterations[view_name])))
        model = XGBClassifier(**params)
        model.fit(train_views[view_name], y_train, verbose=False)
        full_models[view_name] = model
        test_base[:, column_index] = model.predict_proba(test_views[view_name])[:, 1]

    test_stack = meta_model.predict_proba(test_base)[:, 1]
    return VariantRunResult(
        variant_name=variant_name,
        variant_label=variant_label,
        train_probabilities=oof_stack,
        test_probabilities=test_stack,
        train_aux={**{_view_probability_name(view_name): oof_base[:, idx] for idx, view_name in enumerate(view_names)}, "StackedProbability": oof_stack},
        test_aux={**{_view_probability_name(view_name): test_base[:, idx] for idx, view_name in enumerate(view_names)}, "StackedProbability": test_stack},
        fold_summary=pd.DataFrame(fold_rows),
        model_bundle={
            "model_type": "esm_stacking",
            "candidate_name": candidate.name,
            "candidate_label": candidate.label,
            "view_names": view_names,
            "base_models": full_models,
            "meta_model": meta_model,
            "mean_best_iterations": {view_name: float(np.mean(best_iterations[view_name])) for view_name in view_names},
        },
    )


def save_model_bundle(model_bundle: dict, thresholds: dict[str, float], path: Path) -> None:
    payload = dict(model_bundle)
    payload["thresholds"] = thresholds
    with path.open("wb") as handle:
        pickle.dump(payload, handle)


In [ ]:
%%writefile src/pipeline.py
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import matplotlib
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve, average_precision_score

from .data import load_datasets, modification_feature_columns, write_audit
from .features import (
    ESMEmbedder,
    build_esm_feature_views,
    build_handcrafted_features,
    select_handcrafted_features,
)
from .model import (
    THRESHOLD_LABELS,
    build_metric_tables,
    esm_tuning_candidates,
    save_model_bundle,
    train_esm_stacking_variant,
    train_single_xgb_variant,
)


matplotlib.use("Agg")
import matplotlib.pyplot as plt


VARIANT_LABELS = {
    "esm_only": "ESM-only stacking",
    "handcrafted_no_fs": "ESM2 + AAC + DPC + aaIndex without feature selection",
    "handcrafted_with_fs": "ESM2 + AAC + DPC + aaIndex with feature selection",
}


@dataclass
class PipelineConfig:
    project_root: Path
    train_csv: Path
    test_csv: Path
    results_dir: Path
    cache_dir: Path
    model_dir: Path
    variant: str = "esm_only"
    esm_model: str = "esm2_t33_650M_UR50D"
    batch_size: int = 32
    n_splits: int = 5
    seed: int = 42


def _resolve_output_path(path: Path) -> Path:
    try:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("a", encoding="utf-8"):
            pass
        return path
    except PermissionError:
        candidate = path.with_name(f"{path.stem}_updated{path.suffix}")
        suffix_index = 2
        while candidate.exists():
            candidate = path.with_name(f"{path.stem}_updated_{suffix_index}{path.suffix}")
            suffix_index += 1
        return candidate


def _write_csv(df: pd.DataFrame, path: Path, index: bool = False) -> Path:
    target = _resolve_output_path(path)
    df.to_csv(target, index=index)
    return target


def _write_text(content: str, path: Path) -> Path:
    target = _resolve_output_path(path)
    target.write_text(content, encoding="utf-8")
    return target


def _save_figure(fig, path: Path, **kwargs) -> Path:
    target = _resolve_output_path(path)
    fig.savefig(target, **kwargs)
    return target


def _plot_curves(train_labels: np.ndarray, train_prob: np.ndarray, test_labels: np.ndarray, test_prob: np.ndarray, output_path: Path, title_prefix: str) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    train_fpr, train_tpr, _ = roc_curve(train_labels, train_prob)
    test_fpr, test_tpr, _ = roc_curve(test_labels, test_prob)
    axes[0].plot(train_fpr, train_tpr, label=f"Train OOF (AUC={roc_auc_score(train_labels, train_prob):.3f})")
    axes[0].plot(test_fpr, test_tpr, label=f"Independent test (AUC={roc_auc_score(test_labels, test_prob):.3f})")
    axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
    axes[0].set_title(f"{title_prefix} ROC")
    axes[0].set_xlabel("False positive rate")
    axes[0].set_ylabel("True positive rate")
    axes[0].legend()

    train_precision, train_recall, _ = precision_recall_curve(train_labels, train_prob)
    test_precision, test_recall, _ = precision_recall_curve(test_labels, test_prob)
    axes[1].plot(
        train_recall,
        train_precision,
        label=f"Train OOF (AP={average_precision_score(train_labels, train_prob):.3f})",
    )
    axes[1].plot(
        test_recall,
        test_precision,
        label=f"Independent test (AP={average_precision_score(test_labels, test_prob):.3f})",
    )
    axes[1].set_title(f"{title_prefix} Precision-Recall")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].legend()

    fig.tight_layout()
    _save_figure(fig, output_path, dpi=150)
    plt.close(fig)


def _plot_confusion_matrices(
    train_labels: np.ndarray,
    train_prob: np.ndarray,
    test_labels: np.ndarray,
    test_prob: np.ndarray,
    threshold: float,
    threshold_label: str,
    output_path: Path,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_pred = (train_prob >= threshold).astype(int)
    test_pred = (test_prob >= threshold).astype(int)
    train_cm = confusion_matrix(train_labels, train_pred, labels=[0, 1])
    test_cm = confusion_matrix(test_labels, test_pred, labels=[0, 1])

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for axis, matrix, title in (
        (axes[0], train_cm, "Train OOF"),
        (axes[1], test_cm, "Independent test"),
    ):
        axis.imshow(matrix, interpolation="nearest", cmap="Blues")
        axis.set_title(f"{title}\n{threshold_label}")
        axis.set_xticks([0, 1])
        axis.set_xticklabels(["Pred 0", "Pred 1"])
        axis.set_yticks([0, 1])
        axis.set_yticklabels(["True 0", "True 1"])
        for row in range(matrix.shape[0]):
            for col in range(matrix.shape[1]):
                color = "white" if matrix[row, col] > matrix.max() / 2 else "black"
                axis.text(col, row, f"{matrix[row, col]}", ha="center", va="center", color=color)
    fig.tight_layout()
    _save_figure(fig, output_path, dpi=150)
    plt.close(fig)

    train_df = pd.DataFrame(train_cm, index=["TrueNegativeClass", "TruePositiveClass"], columns=["Predicted0", "Predicted1"])
    test_df = pd.DataFrame(test_cm, index=["TrueNegativeClass", "TruePositiveClass"], columns=["Predicted0", "Predicted1"])
    return train_df, test_df


def _select_top_short_peptides(df: pd.DataFrame, probabilities: np.ndarray, max_length: int, top_k: int) -> tuple[pd.DataFrame, dict]:
    ranking = df.loc[:, ["Sequence", "BaseSeq", "Annotation", "FinalLabel", "SeqLength"]].copy()
    ranking["PrimaryProbability"] = probabilities
    ranking = ranking.sort_values("PrimaryProbability", ascending=False).reset_index(drop=True)

    selected_rows = []
    skipped = 0
    for _, row in ranking.iterrows():
        if int(row["SeqLength"]) > max_length:
            skipped += 1
            continue
        selected_rows.append(row.to_dict())
        if len(selected_rows) == top_k:
            break

    return pd.DataFrame(selected_rows), {
        "MaxLength": max_length,
        "RequestedTopK": top_k,
        "SelectedCount": len(selected_rows),
        "SkippedBecauseTooLong": skipped,
    }


def _compose_sequence(base_seq: str, annotation: str) -> str:
    return f"{base_seq} + {annotation}" if annotation else base_seq


def _mutate_sequence(sequence: str, position: int) -> tuple[str, str]:
    replacement = "A" if sequence[position] != "A" else "G"
    mutated = sequence[:position] + replacement + sequence[position + 1 :]
    return mutated, replacement


def _score_esm_rows(df: pd.DataFrame, embedder: ESMEmbedder, model_bundle: dict) -> np.ndarray:
    embeddings = embedder.embed_uncached(df["BaseSeq"].tolist())
    views = build_esm_feature_views(df, embeddings)
    view_names = model_bundle["view_names"]
    base_probabilities = np.column_stack(
        [
            model_bundle["base_models"][view_name].predict_proba(views[view_name])[:, 1]
            for view_name in view_names
        ]
    )
    return model_bundle["meta_model"].predict_proba(base_probabilities)[:, 1]


def _build_attribution_rows(source_row: pd.Series) -> pd.DataFrame:
    rows = []
    original_seq = str(source_row["BaseSeq"])
    annotation = str(source_row["Annotation"] or "")
    mod_columns = modification_feature_columns()
    for position in range(len(original_seq)):
        mutated_seq, mutated_residue = _mutate_sequence(original_seq, position)
        row = {column: source_row[column] for column in mod_columns}
        row.update(
            {
                "Sequence": _compose_sequence(mutated_seq, annotation),
                "BaseSeq": mutated_seq,
                "Annotation": annotation,
                "FinalLabel": int(source_row["FinalLabel"]),
                "SeqLength": len(mutated_seq),
                "Position1Based": position + 1,
                "OriginalResidue": original_seq[position],
                "MutatedResidue": mutated_residue,
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


def _write_attribution_artifacts(
    variant_dir: Path,
    train_top_hits: pd.DataFrame,
    test_top_hits: pd.DataFrame,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    embedder: ESMEmbedder,
    model_bundle: dict,
) -> None:
    heatmaps: list[tuple[str, pd.DataFrame, str, float]] = []
    summary: dict[str, dict] = {}
    for split_name, top_hits, source_df in (
        ("TrainOOF", train_top_hits, train_df),
        ("IndependentTest", test_top_hits, test_df),
    ):
        if top_hits.empty:
            continue
        anchor_hit = top_hits.iloc[0]
        source_match = source_df.loc[
            (source_df["Sequence"] == anchor_hit["Sequence"])
            & (source_df["BaseSeq"] == anchor_hit["BaseSeq"])
            & (source_df["Annotation"] == anchor_hit["Annotation"])
        ]
        anchor_row = source_match.iloc[0] if not source_match.empty else source_df.loc[source_df["BaseSeq"] == anchor_hit["BaseSeq"]].iloc[0]
        mutated_rows = _build_attribution_rows(anchor_row)
        mutated_probabilities = _score_esm_rows(mutated_rows, embedder, model_bundle)
        original_probability = float(anchor_hit["PrimaryProbability"])
        mutated_rows["OriginalProbability"] = original_probability
        mutated_rows["MutatedProbability"] = mutated_probabilities
        mutated_rows["ProbabilityDrop"] = original_probability - mutated_probabilities
        _write_csv(mutated_rows, variant_dir / f"{split_name.lower()}_attribution.csv", index=False)
        heatmaps.append(
            (
                split_name,
                mutated_rows,
                str(anchor_hit["BaseSeq"]),
                original_probability,
            )
        )
        summary[split_name] = {
            "Peptide": str(anchor_hit["BaseSeq"]),
            "OriginalProbability": original_probability,
            "Length": int(anchor_hit["SeqLength"]),
            "MaxProbabilityDrop": float(mutated_rows["ProbabilityDrop"].max()),
            "MeanProbabilityDrop": float(mutated_rows["ProbabilityDrop"].mean()),
        }

    if not heatmaps:
        return

    fig, axes = plt.subplots(len(heatmaps), 1, figsize=(14, 2.8 * len(heatmaps)))
    if len(heatmaps) == 1:
        axes = [axes]

    for axis, (split_name, mutated_rows, peptide, original_probability) in zip(axes, heatmaps):
        matrix = mutated_rows["ProbabilityDrop"].to_numpy(dtype=float).reshape(1, -1)
        image = axis.imshow(matrix, aspect="auto", cmap="coolwarm")
        axis.set_title(f"{split_name} attribution for {peptide} (score={original_probability:.3f})")
        axis.set_yticks([])
        axis.set_xticks(range(len(mutated_rows)))
        axis.set_xticklabels(
            [
                f"{row.Position1Based}\n{row.OriginalResidue}->{row.MutatedResidue}"
                for row in mutated_rows.itertuples()
            ],
            fontsize=8,
        )
        for column_index, value in enumerate(mutated_rows["ProbabilityDrop"].tolist()):
            axis.text(column_index, 0, f"{value:.2f}", ha="center", va="center", fontsize=8, color="black")
    fig.colorbar(image, ax=axes, orientation="vertical", fraction=0.02, pad=0.02, label="Probability drop")
    fig.tight_layout()
    _save_figure(fig, variant_dir / "attribution_heatmaps.png", dpi=150)
    plt.close(fig)
    _write_text(json.dumps(summary, indent=2), variant_dir / "attribution_summary.json")


def _write_variant_predictions(
    variant_dir: Path,
    variant_name: str,
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    train_prob: np.ndarray,
    test_prob: np.ndarray,
    train_aux: dict[str, np.ndarray],
    test_aux: dict[str, np.ndarray],
    thresholds: dict[str, float],
) -> None:
    train_predictions = train_df.loc[:, ["Sequence", "BaseSeq", "Annotation", "SeqLength", "FinalLabel"]].copy()
    train_predictions["VariantName"] = variant_name
    for column_name, values in train_aux.items():
        train_predictions[column_name] = values
    train_predictions["PrimaryProbability"] = train_prob
    _write_csv(train_predictions, variant_dir / "train_oof_predictions.csv", index=False)

    test_predictions = test_df.loc[:, ["Sequence", "BaseSeq", "Annotation", "SeqLength", "FinalLabel"]].copy()
    test_predictions["VariantName"] = variant_name
    for column_name, values in test_aux.items():
        test_predictions[column_name] = values
    test_predictions["PrimaryProbability"] = test_prob
    for threshold_key, threshold_value in thresholds.items():
        readable_label = THRESHOLD_LABELS[threshold_key].replace(" ", "")
        test_predictions[f"PredictedLabel_{readable_label}"] = (test_prob >= threshold_value).astype(int)
    _write_csv(test_predictions, variant_dir / "test_predictions.csv", index=False)


def _write_comparison(metrics_tables: list[pd.DataFrame], results_dir: Path) -> None:
    all_metrics = pd.concat(metrics_tables, ignore_index=True)
    _write_csv(all_metrics, results_dir / "model_comparison_full_metrics.csv", index=False)
    comparison = (
        all_metrics.loc[
            (all_metrics["EvaluationSplit"] == "IndependentTest")
            & (all_metrics["ThresholdKey"] == "best_mcc_on_train_oof"),
            [
                "VariantName",
                "VariantLabel",
                "ThresholdValue",
                "RocAuc",
                "PrecisionRecallAuc",
                "MatthewsCorrcoef",
                "BalancedAccuracy",
                "Precision",
                "RecallSensitivity",
                "Specificity",
                "F1Score",
                "F2Score",
            ],
        ]
        .sort_values(["MatthewsCorrcoef", "PrecisionRecallAuc"], ascending=False)
        .reset_index(drop=True)
    )
    _write_csv(comparison, results_dir / "model_comparison_summary.csv", index=False)


def _combined_esm_handcrafted_features(
    train_embeddings,
    test_embeddings,
    train_handcrafted,
    test_handcrafted,
) -> tuple[np.ndarray, np.ndarray, list[str]]:
    train_features = np.concatenate(
        [
            train_embeddings.mean_embeddings,
            train_embeddings.max_embeddings,
            train_handcrafted.features,
        ],
        axis=1,
    ).astype(np.float32)
    test_features = np.concatenate(
        [
            test_embeddings.mean_embeddings,
            test_embeddings.max_embeddings,
            test_handcrafted.features,
        ],
        axis=1,
    ).astype(np.float32)
    feature_names = (
        [f"ESMMean_{index:04d}" for index in range(train_embeddings.mean_embeddings.shape[1])]
        + [f"ESMMax_{index:04d}" for index in range(train_embeddings.max_embeddings.shape[1])]
        + train_handcrafted.feature_names
    )
    return train_features, test_features, feature_names


def _run_single_variant(config: PipelineConfig, bundle, variant_name: str) -> pd.DataFrame:
    variant_dir = config.results_dir / variant_name
    variant_dir.mkdir(parents=True, exist_ok=True)
    y_train = bundle.train_df["FinalLabel"].to_numpy(dtype=int)
    y_test = bundle.test_df["FinalLabel"].to_numpy(dtype=int)

    run_metadata = {
        "VariantName": variant_name,
        "VariantLabel": VARIANT_LABELS[variant_name],
        "Seed": config.seed,
        "NumFolds": config.n_splits,
    }

    embedder = ESMEmbedder(model_name=config.esm_model, batch_size=config.batch_size)
    train_embeddings = embedder.embed(
        bundle.train_df["BaseSeq"].tolist(),
        config.cache_dir / "shared_train_embeddings.npz",
    )
    test_embeddings = embedder.embed(
        bundle.test_df["BaseSeq"].tolist(),
        config.cache_dir / "shared_test_embeddings.npz",
    )
    run_metadata["EmbeddingModel"] = config.esm_model
    run_metadata["EmbeddingBatchSize"] = config.batch_size
    run_metadata["EmbeddingDevice"] = embedder.device

    if variant_name == "esm_only":
        train_views = build_esm_feature_views(bundle.train_df, train_embeddings)
        test_views = build_esm_feature_views(bundle.test_df, test_embeddings)
        tuning_records = []
        candidate_results = []
        for candidate in esm_tuning_candidates():
            candidate_result = train_esm_stacking_variant(
                variant_name=variant_name,
                variant_label=VARIANT_LABELS[variant_name],
                train_views=train_views,
                test_views=test_views,
                y_train=y_train,
                seed=config.seed,
                n_splits=config.n_splits,
                candidate=candidate,
            )
            candidate_metrics, candidate_thresholds = build_metric_tables(
                variant_name=variant_name,
                variant_label=VARIANT_LABELS[variant_name],
                y_train=y_train,
                y_test=y_test,
                train_prob=candidate_result.train_probabilities,
                test_prob=candidate_result.test_probabilities,
            )
            train_row = candidate_metrics.loc[
                (candidate_metrics["EvaluationSplit"] == "TrainOOF")
                & (candidate_metrics["ThresholdKey"] == "best_mcc_on_train_oof")
            ].iloc[0]
            test_row = candidate_metrics.loc[
                (candidate_metrics["EvaluationSplit"] == "IndependentTest")
                & (candidate_metrics["ThresholdKey"] == "best_mcc_on_train_oof")
            ].iloc[0]
            tuning_records.append(
                {
                    "CandidateName": candidate.name,
                    "CandidateLabel": candidate.label,
                    "Views": ",".join(candidate.view_names),
                    "TrainOofPrecisionRecallAuc": float(train_row["PrecisionRecallAuc"]),
                    "TrainOofMatthewsCorrcoef": float(train_row["MatthewsCorrcoef"]),
                    "IndependentTestPrecisionRecallAuc": float(test_row["PrecisionRecallAuc"]),
                    "IndependentTestMatthewsCorrcoef": float(test_row["MatthewsCorrcoef"]),
                    "ChosenThresholdValue": float(candidate_thresholds["best_mcc_on_train_oof"]),
                }
            )
            candidate_results.append((candidate, candidate_result, candidate_metrics))

        tuning_summary = pd.DataFrame(tuning_records).sort_values(
            ["TrainOofPrecisionRecallAuc", "TrainOofMatthewsCorrcoef"],
            ascending=False,
        ).reset_index(drop=True)
        chosen_candidate_name = str(tuning_summary.iloc[0]["CandidateName"])
        chosen_candidate, result, _ = next(
            entry for entry in candidate_results if entry[0].name == chosen_candidate_name
        )
        _write_csv(tuning_summary, variant_dir / "tuning_summary.csv", index=False)
        run_metadata["TuningCandidates"] = tuning_records
        run_metadata["ChosenCandidate"] = {
            "CandidateName": chosen_candidate.name,
            "CandidateLabel": chosen_candidate.label,
            "Views": list(chosen_candidate.view_names),
        }
    else:
        train_handcrafted = build_handcrafted_features(
            bundle.train_df,
            config.cache_dir / "handcrafted_train_features.npz",
        )
        test_handcrafted = build_handcrafted_features(
            bundle.test_df,
            config.cache_dir / "handcrafted_test_features.npz",
        )
        train_features, test_features, feature_names = _combined_esm_handcrafted_features(
            train_embeddings=train_embeddings,
            test_embeddings=test_embeddings,
            train_handcrafted=train_handcrafted,
            test_handcrafted=test_handcrafted,
        )
        if variant_name == "handcrafted_with_fs":
            selection = select_handcrafted_features(
                train_features=train_features,
                test_features=test_features,
                y_train=y_train,
                feature_names=feature_names,
                seed=config.seed,
            )
            train_features = selection.train_features
            test_features = selection.test_features
            feature_names = selection.feature_names
            _write_text(json.dumps(selection.selection_summary, indent=2), variant_dir / "feature_selection_summary.json")
            run_metadata["FeatureSelection"] = selection.selection_summary
        else:
            run_metadata["FeatureSelection"] = "disabled"

        result = train_single_xgb_variant(
            variant_name=variant_name,
            variant_label=VARIANT_LABELS[variant_name],
            train_features=train_features,
            test_features=test_features,
            y_train=y_train,
            seed=config.seed,
            n_splits=config.n_splits,
            feature_names=feature_names,
        )
        run_metadata["FeatureCount"] = len(feature_names)
        run_metadata["FeatureComposition"] = {
            "EsmMeanDimensions": int(train_embeddings.mean_embeddings.shape[1]),
            "EsmMaxDimensions": int(train_embeddings.max_embeddings.shape[1]),
            "HandcraftedDimensions": int(train_handcrafted.features.shape[1]),
        }

    metrics_table, thresholds = build_metric_tables(
        variant_name=result.variant_name,
        variant_label=result.variant_label,
        y_train=y_train,
        y_test=y_test,
        train_prob=result.train_probabilities,
        test_prob=result.test_probabilities,
    )
    _write_csv(metrics_table, variant_dir / "metrics.csv", index=False)
    _write_csv(result.fold_summary, variant_dir / "fold_summary.csv", index=False)
    _write_variant_predictions(
        variant_dir=variant_dir,
        variant_name=result.variant_name,
        train_df=bundle.train_df,
        test_df=bundle.test_df,
        train_prob=result.train_probabilities,
        test_prob=result.test_probabilities,
        train_aux=result.train_aux,
        test_aux=result.test_aux,
        thresholds=thresholds,
    )

    confusion_threshold_key = "best_mcc_on_train_oof"
    confusion_threshold = thresholds[confusion_threshold_key]
    train_cm_df, test_cm_df = _plot_confusion_matrices(
        train_labels=y_train,
        train_prob=result.train_probabilities,
        test_labels=y_test,
        test_prob=result.test_probabilities,
        threshold=confusion_threshold,
        threshold_label=THRESHOLD_LABELS[confusion_threshold_key],
        output_path=variant_dir / "confusion_matrices.png",
    )
    _write_csv(train_cm_df, variant_dir / "train_confusion_matrix.csv", index=True)
    _write_csv(test_cm_df, variant_dir / "test_confusion_matrix.csv", index=True)

    train_top_hits, train_top_summary = _select_top_short_peptides(
        bundle.train_df,
        result.train_probabilities,
        max_length=15,
        top_k=10,
    )
    test_top_hits, test_top_summary = _select_top_short_peptides(
        bundle.test_df,
        result.test_probabilities,
        max_length=15,
        top_k=10,
    )
    _write_csv(train_top_hits, variant_dir / "train_top10_short_peptides.csv", index=False)
    _write_csv(test_top_hits, variant_dir / "test_top10_short_peptides.csv", index=False)
    _write_text(
        json.dumps({"TrainOOF": train_top_summary, "IndependentTest": test_top_summary}, indent=2),
        variant_dir / "top_peptide_summary.json",
    )
    if variant_name == "esm_only":
        _write_attribution_artifacts(
            variant_dir=variant_dir,
            train_top_hits=train_top_hits,
            test_top_hits=test_top_hits,
            train_df=bundle.train_df,
            test_df=bundle.test_df,
            embedder=embedder,
            model_bundle=result.model_bundle,
        )

    _plot_curves(
        train_labels=y_train,
        train_prob=result.train_probabilities,
        test_labels=y_test,
        test_prob=result.test_probabilities,
        output_path=variant_dir / "roc_pr_curves.png",
        title_prefix=VARIANT_LABELS[variant_name],
    )
    save_model_bundle(result.model_bundle, thresholds, config.model_dir / f"{variant_name}_bundle.pkl")
    _write_text(json.dumps(run_metadata, indent=2), variant_dir / "run_config.json")
    return metrics_table


def run_pipeline(config: PipelineConfig) -> None:
    bundle = load_datasets(config.train_csv, config.test_csv)
    write_audit(bundle.audit, config.results_dir / "data_audit.json")

    if config.variant == "all":
        variant_names = list(VARIANT_LABELS.keys())
    else:
        variant_names = [config.variant]

    metrics_tables = []
    for variant_name in variant_names:
        metrics_tables.append(_run_single_variant(config, bundle, variant_name))

    _write_comparison(metrics_tables, config.results_dir)


In [ ]:
%%writefile main.py
from __future__ import annotations

import argparse
from pathlib import Path

from src.pipeline import PipelineConfig, run_pipeline


def build_parser() -> argparse.ArgumentParser:
    root = Path(__file__).resolve().parent
    parser = argparse.ArgumentParser(
        description="IL-10 peptide training pipeline with switchable ESM-only and handcrafted variants."
    )
    parser.add_argument(
        "--train-csv",
        type=Path,
        default=root / "data" / "peptide_level_dataset_MHCII.csv",
    )
    parser.add_argument(
        "--test-csv",
        type=Path,
        default=root / "data" / "all_minus_benchmark_minus_mhcii.csv",
    )
    parser.add_argument(
        "--results-dir",
        type=Path,
        default=root / "results",
    )
    parser.add_argument(
        "--cache-dir",
        type=Path,
        default=root / "artifacts" / "cache",
    )
    parser.add_argument(
        "--model-dir",
        type=Path,
        default=root / "artifacts" / "models",
    )
    parser.add_argument(
        "--esm-model",
        default="esm2_t33_650M_UR50D",
    )
    parser.add_argument(
        "--variant",
        choices=["esm_only", "handcrafted_no_fs", "handcrafted_with_fs", "all"],
        default="esm_only",
        help="Choose one model variant or run all variants for comparison.",
    )
    parser.add_argument(
        "--batch-size",
        type=int,
        default=32,
    )
    parser.add_argument(
        "--num-folds",
        type=int,
        default=5,
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
    )
    return parser


def main() -> None:
    parser = build_parser()
    args = parser.parse_args()
    args.results_dir.mkdir(parents=True, exist_ok=True)
    args.cache_dir.mkdir(parents=True, exist_ok=True)
    args.model_dir.mkdir(parents=True, exist_ok=True)

    config = PipelineConfig(
        project_root=Path(__file__).resolve().parent,
        train_csv=args.train_csv,
        test_csv=args.test_csv,
        results_dir=args.results_dir,
        cache_dir=args.cache_dir,
        model_dir=args.model_dir,
        variant=args.variant,
        esm_model=args.esm_model,
        batch_size=args.batch_size,
        n_splits=args.num_folds,
        seed=args.seed,
    )
    run_pipeline(config)


if __name__ == "__main__":
    main()


## 3. 寫入內嵌資料

兩個 CSV 已用 base64 內嵌在這個 notebook（約 100 KB）。執行此格會自動解碼寫入 `data/`，**完全不需要上傳**。


In [ ]:
import base64, os
os.makedirs('data', exist_ok=True)
_FILES = {
    'peptide_level_dataset_MHCII.csv': "...",
    'all_minus_benchmark_minus_mhcii.csv': "...",
}
for _name, _b64 in _FILES.items():
    with open(os.path.join('data', _name), 'wb') as _f:
        _f.write(base64.b64decode(_b64))
print('data/ 內容：', os.listdir('data'))


## 4. 訓練

預設只跑 **`esm_only`**（效果最好的版本）。
- 想跑三版本比較：把 `variant` 改成 `"all"`。
- 想快速冒煙測試：把 `esm_model` 改成 `"esm2_t12_35M_UR50D"`（較小、較快）。


In [ ]:
import torch
# 相容性修正：較新版 PyTorch 的 torch.load 預設 weights_only=True，
# 會讓 fair-esm 載入官方權重失敗。ESM-2 權重來自官方來源，這裡明確設回 False。
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_load(*args, **kwargs)
torch.load = _safe_load

from pathlib import Path
from src.pipeline import PipelineConfig, run_pipeline

config = PipelineConfig(
    project_root=Path('.'),
    train_csv=Path('data/peptide_level_dataset_MHCII.csv'),
    test_csv=Path('data/all_minus_benchmark_minus_mhcii.csv'),
    results_dir=Path('results'),
    cache_dir=Path('artifacts/cache'),
    model_dir=Path('artifacts/models'),
    variant='esm_only',                 # 想比較三版本 → 'all'
    esm_model='esm2_t33_650M_UR50D',    # 快速測試 → 'esm2_t12_35M_UR50D'
    batch_size=32,
    n_splits=5,
    seed=42,
)
run_pipeline(config)
print('✅ 完成，結果寫入 results/')


## 5. 檢視結果

In [ ]:
import pandas as pd
from IPython.display import Image, display

metrics = pd.read_csv('results/esm_only/metrics.csv')
display(metrics)

for img in ['results/esm_only/roc_pr_curves.png',
            'results/esm_only/confusion_matrices.png',
            'results/esm_only/attribution_heatmaps.png']:
    try:
        display(Image(img))
    except FileNotFoundError:
        pass

print('\nTest 集前 10 條短胜肽：')
display(pd.read_csv('results/esm_only/test_top10_short_peptides.csv'))


## 6. 下載結果（可選）

In [ ]:
!zip -q -r results.zip results artifacts/models
from google.colab import files
files.download('results.zip')
